# Correcting data types

Correcting data types is one of the **critical parts of data engineering**: it is directly related to the schema.
A column with the wrong type can be stored fine, but it gives wrong results (or errors) as soon as we calculate with it.

In notebook 07 we loaded the flight data into the table `dev.spark_db.flight_time_raw` with the right schema **for the file**.
Here we look at that table for analysis, fix the types of the time columns, and save the result in a new table, `dev.spark_db.flight_time`.

### Steps
1. Read the table into a DataFrame
2. Convert **one** time column, step by step, to see how it works
3. Put that logic in a function and apply it to **all** the time columns
4. Save the result into a new table and check it

In [10]:
%sql
-- Look at the raw table: the time columns are plain numbers (bigint / int)
select * from dev.spark_db.flight_time_raw

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
0,2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",1115,1113.0,1343.0,5.0,1400,1348.0,0,946
1,2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",1315,1311.0,1536.0,7.0,1559,1543.0,0,946
2,2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",1415,1414.0,1642.0,9.0,1721,1651.0,0,946
3,2000-01-01,DL,1997,BOS,"Boston, MA",ATL,"Atlanta, GA",1715,1720.0,1955.0,10.0,2013,2005.0,0,946
4,2000-01-01,DL,2065,BOS,"Boston, MA",ATL,"Atlanta, GA",2015,2010.0,2230.0,10.0,2300,2240.0,0,946
5,2000-01-01,US,2619,BOS,"Boston, MA",ATL,"Atlanta, GA",650,649.0,956.0,7.0,955,1003.0,0,946
6,2000-01-01,US,2621,BOS,"Boston, MA",ATL,"Atlanta, GA",1440,1446.0,1713.0,4.0,1738,1717.0,0,946
7,2000-01-01,DL,346,BTR,"Baton Rouge, LA",ATL,"Atlanta, GA",1740,1744.0,1957.0,9.0,2008,2006.0,0,449
8,2000-01-01,DL,412,BTR,"Baton Rouge, LA",ATL,"Atlanta, GA",1345,1345.0,1552.0,9.0,1622,1601.0,0,449
9,2000-01-01,DL,299,BUF,"Buffalo, NY",ATL,"Atlanta, GA",1245,1245.0,1443.0,5.0,1455,1448.0,0,712


## The problem: time columns stored as numbers

After looking at the table, these columns store times of day as plain numbers in `hhmm` format (for example `1430` means 14:30):

`CRS_DEP_TIME`, `DEP_TIME`, `WHEELS_ON`, `CRS_ARR_TIME`, `ARR_TIME`

As numbers, they can't be used for time calculations: `1430 - 1355` gives `75`, but the real difference is 35 minutes.
To analyse them properly, we convert each one to an **`INTERVAL HOUR TO MINUTE`** value (hours and minutes, the time since midnight).

`TAXI_IN` is different: it is not a time of day but a **duration in minutes** (for example `7`), so it becomes an **`INTERVAL MINUTE`**.

## Step 1. Read the table into a DataFrame

The first step for any transformation is to read the data from the table and bring it into a DataFrame,
so it is ready for processing. (As always, this is lazy: only the schema is read here.)

In [11]:
# Read the table into a DataFrame (schema comes from the table, no inference)
flight_time_raw_df = spark.read.table("dev.spark_db.flight_time_raw")

## Step 2. Convert one column, step by step

We start with `CRS_DEP_TIME` only, and keep the intermediate columns so we can see each step:

| Step | Expression | `905` becomes |
|---|---|---|
| Pad to 4 digits with `0` in front | `lpad(CRS_DEP_TIME, 4, '0')` | `"0905"` |
| Hours: first 2 characters | `left(..., 2)` | `"09"` |
| Minutes: last 2 characters | `right(..., 2)` | `"05"` |
| Glue them with `:` | `concat(hh, ':', mm)` | `"09:05"` |
| Turn the text into an interval | `cast(... AS INTERVAL HOUR TO MINUTE)` | 9 hours 5 minutes |

Why `lpad` first? Times before 10:00 have only 3 digits (`905`). Without padding, `left(905, 2)` would give `"90"`.

Methods used:
- `withColumns({"name": column, ...})`: add or replace **several columns at once**, keep all the other columns
- `expr("sql expression")`: write the column's logic as a SQL expression inside Python

In [ ]:
from pyspark.sql.functions import expr

# withColumns() is a method in PySpark that allows you to add new columns or modify existing columns in a DataFrame. It takes one or more column expressions as arguments and returns a new DataFrame with the specified changes.
# in case of a data frame with 10 columns and you want to change the data type of 5 columns you can use withColumns() method to change the data type of those 5 columns and keep the other 5 columns as it is.
# the expr method is used to create a new column or modify an existing column in a DataFrame using a SQL expression. It takes a string argument that represents the SQL expression and returns a new column with the specified changes.
# lpad is used to left pad a string with a specified character to a specified length. It takes three arguments: the string to be padded, the length to pad to, and the character to pad with. If the string is already longer than the specified length, it will be truncated.
# left is used to extract a specified number of characters from the left side of a string. It takes two arguments: the string to extract from and the number of characters to extract. If the string is shorter than the specified number of characters, it will return the entire string.
# right works the same way as left, but takes the characters from the right side.

# Step 2a: split CRS_DEP_TIME (e.g. 905) into hours ("09") and minutes ("05")
step1_df = (
    flight_time_raw_df.withColumns({
        "CRS_DEP_TIME_HH": expr("left(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
        "CRS_DEP_TIME_MM": expr("right(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
    })
)

# the cast function is used to convert a column from one data type to another. It takes a string argument that represents the target data type and returns a new column with the specified data type.
# Step 2b: glue them as "09:05" and cast the text to an interval (9 hours 5 minutes)
step2_df = (
   step1_df.withColumns({
        "CRS_DEP_TIME_NEW": expr("cast(concat(CRS_DEP_TIME_HH, ':', CRS_DEP_TIME_MM) AS INTERVAL HOUR TO MINUTE)"),
   })
)

# Action: show 3 rows to check the new columns
# (outside Databricks, use step2_df.select("CRS_DEP_TIME", "CRS_DEP_TIME_NEW").show(3))
display(step2_df.limit(3))

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE,CRS_DEP_TIME_HH,CRS_DEP_TIME_MM,CRS_DEP_TIME_NEW
0,2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",1115,1113,1343,5,1400,1348,0,946,11,15,0 days 11:15:00
1,2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",1315,1311,1536,7,1559,1543,0,946,13,15,0 days 13:15:00
2,2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",1415,1414,1642,9,1721,1651,0,946,14,15,0 days 14:15:00


## Step 3. Apply it to all the time columns with a function

Writing these expressions again for every column would be long and easy to get wrong.
So we put the logic in a Python function, `get_interval(column_name)`, that **returns the column expression**,
and call it once per column.

Two things to notice:
- We give `withColumns` the **same names** as the existing columns, so the converted values **replace** the old ones in place.
  No need to drop the old column and rename the new one, and the column order stays the same as the table.
- `TAXI_IN` doesn't need the function: it is already a number of minutes, so a simple `cast(TAXI_IN AS INTERVAL MINUTE)` is enough.

In [24]:
# Now we need CRS_DEP_TIME_NEW INSTEAD of CRS_DEP_TIME in the final table.
# Instead of dropping the old column and renaming the new one, we give withColumns()
# the SAME column name: it then replaces that column in place (same position, new values).
# And we need to do it for the rest of the columns: DEP_TIME, WHEELS_ON, CRS_ARR_TIME, ARR_TIME,
# so we put the logic in a function and don't have to write the same code for each column.

# Goal: turn a time stored as a number (1430) into a real time (14:30)

def get_interval(hhmm_value):
    # Why only 1 argument?
    # → We pass ONE column name, e.g. "CRS_DEP_TIME".
    #   Its values hold hours + minutes together (1430).
    #   left() and right() below cut it into 2 pieces.

    from pyspark.sql.functions import expr   # lets us write SQL inside Python (already imported above, harmless)

    # Why """ and not "?
    # → In Python, """ lets the SQL text go over many lines, so it's easier to read.
    # Why f"..."?
    # → The f-string puts the column name in place of {hhmm_value}.

    # What the SQL does, with 905 as an example:
    #   lpad   → "0905"   (add a 0 in front, so it's always 4 digits)
    #   left   → "09"     (hours)
    #   right  → "05"     (minutes)
    #   concat → "09:05"  (glue them)
    #   cast   → 9 hours 5 minutes (a real time)

    return expr(f"""
                cast(concat(left(lpad({hhmm_value}, 4, '0'), 2), ':',
                            right(lpad({hhmm_value}, 4, '0'), 2))
                            AS INTERVAL HOUR TO MINUTE)
                """)


# Use it on all 5 time columns (write once, use many times)
# TAXI_IN is not a time of day but a number of minutes (e.g. 7), so it is cast straight to INTERVAL MINUTE
flight_time_df = flight_time_raw_df.withColumns({
    "CRS_DEP_TIME": get_interval("CRS_DEP_TIME"),
    "DEP_TIME":     get_interval("DEP_TIME"),
    "WHEELS_ON":    get_interval("WHEELS_ON"),
    "CRS_ARR_TIME": get_interval("CRS_ARR_TIME"),
    "ARR_TIME":     get_interval("ARR_TIME"),
    "TAXI_IN":      expr("cast(TAXI_IN AS INTERVAL MINUTE)"),
})

## Step 4. Save the results into a new table, `flight_time`

`overwrite` replaces the table's data if it already exists. Saving is an **action**: this is when the whole job runs.

In [27]:
# Write the corrected DataFrame into a new table, replacing any existing data
flight_time_df.write.mode("overwrite").saveAsTable("dev.spark_db.flight_time")

In [28]:
%sql
-- Check the new table: the time columns are now intervals
select * from dev.spark_db.flight_time

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
0,2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 11:15:00,0 days 11:13:00,1343.0,0 days 00:05:00,0 days 14:00:00,0 days 13:48:00,0,946
1,2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 13:15:00,0 days 13:11:00,1536.0,0 days 00:07:00,0 days 15:59:00,0 days 15:43:00,0,946
2,2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 14:15:00,0 days 14:14:00,1642.0,0 days 00:09:00,0 days 17:21:00,0 days 16:51:00,0,946
3,2000-01-01,DL,1997,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 17:15:00,0 days 17:20:00,1955.0,0 days 00:10:00,0 days 20:13:00,0 days 20:05:00,0,946
4,2000-01-01,DL,2065,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 20:15:00,0 days 20:10:00,2230.0,0 days 00:10:00,0 days 23:00:00,0 days 22:40:00,0,946
5,2000-01-01,US,2619,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 06:50:00,0 days 06:49:00,956.0,0 days 00:07:00,0 days 09:55:00,0 days 10:03:00,0,946
6,2000-01-01,US,2621,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 14:40:00,0 days 14:46:00,1713.0,0 days 00:04:00,0 days 17:38:00,0 days 17:17:00,0,946
7,2000-01-01,DL,346,BTR,"Baton Rouge, LA",ATL,"Atlanta, GA",0 days 17:40:00,0 days 17:44:00,1957.0,0 days 00:09:00,0 days 20:08:00,0 days 20:06:00,0,449
8,2000-01-01,DL,412,BTR,"Baton Rouge, LA",ATL,"Atlanta, GA",0 days 13:45:00,0 days 13:45:00,1552.0,0 days 00:09:00,0 days 16:22:00,0 days 16:01:00,0,449
9,2000-01-01,DL,299,BUF,"Buffalo, NY",ATL,"Atlanta, GA",0 days 12:45:00,0 days 12:45:00,1443.0,0 days 00:05:00,0 days 14:55:00,0 days 14:48:00,0,712


## Result: the types before and after

| Column | `flight_time_raw` | `flight_time` |
|---|---|---|
| `CRS_DEP_TIME`, `DEP_TIME`, `CRS_ARR_TIME`, `ARR_TIME` | `bigint` (`1430`) | `interval hour to minute` (14:30) |
| `WHEELS_ON` | `int` (`1430`) | `interval hour to minute` (14:30) |
| `TAXI_IN` | `int` (`7`) | `interval minute` (7 minutes) |
| All the other columns | unchanged | unchanged |

Now time calculations work, for example the departure delay: `DEP_TIME - CRS_DEP_TIME`.